In [6]:
import pandas as pd

In [7]:
us = pd.read_csv("/Users/nic/Documents/MM2/data/us_cases.csv")

us_u = pd.DataFrame({
    "case_id":                us["matter_number"].fillna(us["url"]),
    "jurisdiction":           "US",
    "regulator":              "FTC",
    "decision_date":          pd.to_datetime(us["date"], errors="coerce"),
    "statute_or_article":     us["statutes_final"],
    "penalty_amount":         us["penalty_usd_enriched"],
    "penalty_currency": us["penalty_usd_enriched"].apply(lambda x: "USD" if pd.notna(x) else pd.NA),
    "decision_text":          us["press_release_text"],
    "text_source":            "ftc_press_release",
    "violation_labels_native": us["violation_labels_final"],
    "label_source":           us["label_source"],
})
print(us_u.shape)

(337, 11)


In [8]:
eu = pd.read_csv("/Users/nic/Documents/MM2/data/gdprhub_dpa_decisions.csv")

eu["currency"] = eu["currency"].replace("€", "EUR")
EURO_COUNTRIES = ["Austria", "Belgium", "Croatia", "Cyprus", "Estonia", "Finland",
                  "France", "Germany", "Greece", "Ireland", "Italy", "Latvia",
                  "Lithuania", "Luxembourg", "Malta", "Netherlands", "Portugal",
                  "Slovakia", "Slovenia", "Spain"]
needs_eur = eu["currency"].isna() & eu["fine_numeric"].notna() & eu["jurisdiction"].isin(EURO_COUNTRIES)
eu.loc[needs_eur, "currency"] = "EUR"
print(f"currency imputed EUR for {needs_eur.sum()} Eurozone fines")

def join_text(row):
    parts = [p for p in [row["facts"], row["holding"]] if isinstance(p, str)]
    return "\n\n".join(parts) if parts else None

eu_u = pd.DataFrame({
    "case_id":                eu["title"],
    "jurisdiction":           eu["jurisdiction"],
    "regulator":              eu["authority"],
    "decision_date":          pd.to_datetime(eu["date_parsed"], errors="coerce"),
    "statute_or_article":     eu["gdpr_articles"],
    "penalty_amount":         eu["fine_numeric"],
    "penalty_currency":       eu["currency"],
    "decision_text":          eu.apply(join_text, axis=1),
    "text_source":            "gdprhub_summary",
    "violation_labels_native": eu["gdpr_articles"],
    "label_source":           "gdprhub_categories",
})
print(eu_u.shape)

currency imputed EUR for 32 Eurozone fines
(2417, 11)


In [9]:
unified = pd.concat([us_u, eu_u], ignore_index=True)

qa = unified.groupby("jurisdiction").agg(
    cases=("case_id", "size"),
    with_text=("decision_text", lambda s: s.notna().sum()),
    with_penalty=("penalty_amount", lambda s: s.notna().sum()),
    with_labels=("violation_labels_native", lambda s: s.notna().sum()),
)
print(qa)

unified.to_csv("/Users/nic/Documents/MM2/data/unified_cases.csv", index=False)
print(unified.shape)
unified.sample(3, random_state=1).T

                cases  with_text  with_penalty  with_labels
jurisdiction                                               
Austria           105        105            18           99
Belgium           247        247            46          240
Bulgaria           10         10             7           10
Croatia            27         27            12           26
Cyprus             27         27            10           27
Czech Republic      5          5             4            5
Denmark            95         95             7           95
Estonia            30         30             2           27
European Union     13         13             1           11
Finland            89         89            19           87
France             84         84            65           76
Germany            22         21             8           22
Greece            129        129            84          124
Hungary            60         60            40           60
Iceland           101        101        

,1785,2639,857
case_id,Garante per la protezione dei dati personali (...,Tietosuojavaltuutetun toimisto (Finland) - TSV...,APD/GBA (Belgium) - 02/2021
jurisdiction,Italy,Finland,Belgium
regulator,Garante per la protezione dei dati personali (...,Tietosuojavaltuutetun toimisto (Finland),APD/GBA (Belgium)
decision_date,2025-12-18 00:00:00,2024-12-17 00:00:00,2021-01-12 00:00:00
statute_or_article,Article 4(11) GDPR; Article 5 GDPR; Article 5(...,Article 5(1)(f) GDPR; Article 25(1) GDPR; Arti...,Article 6 GDPR; Article 20 GDPR; Article 21 GDPR
penalty_amount,NaN,950000.0,10000.0
penalty_currency,NaN,EUR,EUR
decision_text,"Jaguar Land Rover Italia S.p.A. (JLRI), the It...","On the 23 December 2022, the data subject file...",After a contractual relationship with the musi...
text_source,gdprhub_summary,gdprhub_summary,gdprhub_summary
violation_labels_native,Article 4(11) GDPR; Article 5 GDPR; Article 5(...,Article 5(1)(f) GDPR; Article 25(1) GDPR; Arti...,Article 6 GDPR; Article 20 GDPR; Article 21 GDPR
